# 08 · LSTM sequence model

**Why try it:** an LSTM reads the raw hourly history (72 h) and the hourly forecast (next 24 h) directly, so it can learn timing (e.g. rain at hour 2 vs hour 22) and shapes the tabular features summarize away. Most soil moisture forecasting papers report an LSTM, so reviewers will expect one.

**Why it may not win:** we have 10 stations and engineered features already encode the physics; gradient boosting is often as good on tabular data of this size. Either outcome is a useful result.

Architecture (`soilcast/seq.py`):
```
past 72 h  [sm, st, air_t, rh, log rain, dsm_1h, missing flag] -> LSTM (2 layers) --+
next 24 h  [log rain, air_t, rh] forecast                     -> LSTM (1 layer)  --+-> MLP -> change at +1..+24 h
SSURGO     [sand, silt, clay, awc, drainage, bulk density]    -> Linear+ReLU     --+
```
Loss: Huber on all 24 horizons (masked where missing); early stopping on 2023; test 2024 to 2025.

**Run modes:** `SMOKE` trains a small model on a subsample (CPU, a few minutes) to prove the pipeline. Set `MODE = "FULL"` on Colab (A100) or the RTX 4080 Super, or run `python scripts/train_lstm.py --device cuda` / `--sweep 20`.

In [ ]:
# Setup: works locally (pip install -e . from the repo root) and on Google Colab
import sys, warnings
warnings.filterwarnings("ignore")
if "google.colab" in sys.modules:
    # Upload soilcast.zip to Google Drive (MyDrive) first
    from google.colab import drive
    drive.mount("/content/drive")
    !unzip -q -o /content/drive/MyDrive/soilcast.zip -d /content
    %cd /content/soilcast
    !pip install -q -e ".[gpu-trees]"
    sys.path.insert(0, "/content/soilcast/src")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from soilcast.config import PROCESSED, RAW, RESULTS
from soilcast.plotting import set_style, model_color, SERIES, MUTED
set_style()
pd.set_option("display.width", 160, "display.max_columns", 30, "display.precision", 3)
RESULTS.mkdir(exist_ok=True)
# Colab: Runtime > Change runtime type > GPU. torch is preinstalled on Colab.
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import os
from soilcast import data
from soilcast.seq import hourly_for_seq, train_lstm
from soilcast.evaluate import sm_metrics
MODE = "FULL" if torch.cuda.is_available() else "SMOKE"
CFG = {"SMOKE": dict(past_len=48, hidden=64, fut_hidden=32, layers=1, epochs=4, batch_size=512, max_train=60_000,
                     train_step_h=3, num_workers=0, patience=3),
       "FULL":  dict(past_len=72, hidden=128, fut_hidden=64, layers=2, epochs=30, batch_size=1024, max_train=None,
                     train_step_h=1, num_workers=0 if os.name == "nt" else 4, patience=5)}[MODE]
print(MODE, CFG)
u, soils, nbm = data.load_uscrn(), data.load_soils(), data.load_forecasts()
h = hourly_for_seq(u, soils, 5)
h_nbm = hourly_for_seq(u[u.time_utc >= "2024-09-01"], soils, 5, forecasts=nbm)

In [ ]:
model, hist, preds, n_train = train_lstm(h, {"perfect": h, "nbm": h_nbm}, **CFG)
print("trained on", n_train, "windows")
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(hist.epoch, hist.valid_rmse_pp_24h, marker="o", color=SERIES[2]); ax.set(xlabel="epoch", ylabel="valid RMSE at +24 h (pp)", title="Validation (2023)");

## Compare with persistence and GBM on the same test windows

In [ ]:
g = pd.read_parquet(RESULTS / "03_test_predictions_5cm.parquet")
m = preds["perfect"].merge(g, on=["station", "time_utc"], suffixes=("_lstm", ""))
cmp = sm_metrics(m.sm_t24, {"persistence": m.sm, "gbm v2 (tabular)": m.pred_gbm_v2, "linear v2": m.pred_linear_v2,
                            f"lstm ({MODE})": m.pred_sm_t24}, m.sm, m.fc_precip_24h)
cmp.to_csv(RESULTS / f"08_lstm_vs_gbm_{MODE.lower()}.csv", index=False)
print(len(m), "shared test windows (perfect forecasts)"); cmp

In [ ]:
# Real NBM forecasts, Apr-Dec 2025 (LSTM only; see notebook 04 for the tabular models on NBM)
pn = preds["nbm"]; pn = pn[pn.time_utc >= "2025-04-01"]
tgt = h[["station", "time_utc", "sm"]].assign(sm_t24=h.groupby("station").sm.shift(-24),
       rain_next24=h.groupby("station").precip.transform(lambda s: s[::-1].rolling(24, min_periods=12).sum()[::-1].shift(-1)))
mn = pn.merge(tgt.drop(columns="sm"), on=["station", "time_utc"]).dropna(subset=["sm_t24"])
sm_metrics(mn.sm_t24, {"persistence": mn.sm, f"lstm ({MODE}), NBM": mn.pred_sm_t24}, mn.sm, mn.rain_next24)

## Notes for the full run
- In `SMOKE` mode the LSTM sees ~60k windows for a few epochs: its numbers are a pipeline check, **not** a fair comparison. Judge the LSTM on the `FULL` run.
- Worth sweeping: past window (24/48/72/168 h), hidden size, layers, dropout, learning rate (`scripts/train_lstm.py --sweep 20`).
- Other sequence options if the LSTM is promising: a Temporal Fusion Transformer, or zero-shot time-series foundation models (TimesFM, Chronos) as a no-training baseline; a NeurIPS 2026 workshop paper found TimesFM strong below 1 day and weak beyond it on soil moisture.
- A fair final comparison uses the same test windows, the same forecast source (perfect and NBM), and bootstrap CIs (notebook 03).